# 03 — Entrenar Modelo 2 (single-label, 5 patógenos)

Cada hoja tiene **una** enfermedad (los datasets vienen clasificados con un solo patógeno por imagen) → problema **single-label** con softmax.

**Clases**: `bacterianas`, `fungicas`, `plagas_insectos`, `roya`, `virales`

## Arquitectura y decisiones de diseño

| Componente | Elección | Por qué |
|---|---|---|
| Backbone | EfficientNetB0 | Compacto (5M params), excelente en imágenes de hojas |
| Cabeza | Dense softmax(5) | Clasificación mutuamente excluyente |
| Loss | CategoricalCrossentropy (label smoothing 0.05) | Estándar single-label, bien calibrado |
| Class weights | Inverse frequency | Compensa clases con menos imágenes |
| EMA | decay=0.9999 | Suaviza pesos al final del fine-tune |

## Pipeline

1. EfficientNetB0 pretrained + cabeza softmax(5)
2. Fase 1: solo cabeza (backbone frozen), LR=1e-3
3. Fase 2: fine-tune últimas 80 capas + CosineDecay + EMA
4. Predicción por `argmax`; evaluación con accuracy y F1 por clase
5. Salida: `model2_pathogen.keras`

Target: accuracy / F1 macro > 95.14% / 0.9472 en test (superar la versión Teachable Machine del paper).

In [ ]:
!pip install -q tensorflow albumentations scikit-learn matplotlib

In [ ]:
import json
from pathlib import Path
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)
print("GPU:", tf.config.list_physical_devices("GPU"))

IMG_SIZE = (224, 224)
BATCH = 16
EPOCHS_P1 = 20
EPOCHS_P2 = 45
LR_P1 = 1e-3
LR_P2 = 1e-5
EMA_DECAY = 0.9999
LABEL_SMOOTH = 0.05

DATA = Path("./splits/clasificacion_patogeno")
OUT = Path("./outputs")
OUT.mkdir(exist_ok=True)

In [ ]:
import time
import json
from pathlib import Path
import numpy as np
import albumentations as A
from PIL import Image
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.efficientnet import preprocess_input

DATA = globals().get("DATA", Path("./splits/clasificacion_patogeno"))
OUT = globals().get("OUT", Path("./outputs"))
OUT.mkdir(exist_ok=True)
IMG_SIZE = globals().get("IMG_SIZE", (224, 224))
BATCH = globals().get("BATCH", 16)

TRAIN_AUG = A.Compose([
    A.Rotate(limit=45, border_mode=0, p=0.7),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=0.25),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.MotionBlur(blur_limit=5, p=1.0),
    ], p=0.2),
    A.GaussNoise(std_range=(0.02, 0.1), p=0.15),
    A.Affine(translate_percent=0.08, scale=(0.88, 1.15), rotate=0, p=0.4),
    A.CoarseDropout(
        num_holes_range=(1, 4),
        hole_height_range=(16, 28),
        hole_width_range=(16, 28),
        fill=0,
        p=0.1,
    ),
])
VAL_AUG = A.Compose([])


def safe_read_image(fp, target_size, retries=3, delay=0.5):
    for attempt in range(retries):
        try:
            return np.array(Image.open(fp).convert("RGB").resize(target_size))
        except (OSError, IOError) as exc:
            if attempt < retries - 1:
                time.sleep(delay)
                continue
            print(f"[warn] saltando {fp}: {exc}")
            return None


class LeafSequence(Sequence):
    def __init__(self, directory, img_size, batch_size, augment, shuffle):
        self.img_size = img_size
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.class_indices = {}
        self.samples = []
        classes = sorted(p.name for p in Path(directory).iterdir() if p.is_dir())
        for idx, name in enumerate(classes):
            self.class_indices[name] = idx
            for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
                for fp in (Path(directory) / name).glob(ext):
                    self.samples.append((str(fp), idx))
        self.n = len(self.samples)
        self.num_classes = len(classes)
        self.labels = np.array([s[1] for s in self.samples])
        if shuffle:
            self._shuffle()

    def _shuffle(self):
        np.random.shuffle(self.samples)
        self.labels = np.array([s[1] for s in self.samples])

    def __len__(self):
        return max(1, self.n // self.batch_size)

    def __getitem__(self, batch_idx):
        batch = self.samples[batch_idx * self.batch_size:(batch_idx + 1) * self.batch_size]
        images, labels = [], []
        for fp, cls in batch:
            img = safe_read_image(fp, self.img_size)
            if img is None:
                continue
            img = TRAIN_AUG(image=img)["image"] if self.augment else VAL_AUG(image=img)["image"]
            images.append(img)
            labels.append(cls)
        if not images:
            placeholder = np.zeros((1, *self.img_size, 3), dtype=np.float32)
            placeholder_label = np.zeros((1, self.num_classes), dtype=np.float32)
            return preprocess_input(placeholder), placeholder_label
        images = np.stack(images).astype(np.float32)
        labels = np.eye(self.num_classes, dtype=np.float32)[labels]
        return preprocess_input(images), labels

    def on_epoch_end(self):
        if self.shuffle:
            self._shuffle()


train_gen = LeafSequence(DATA / "train", IMG_SIZE, BATCH, augment=True, shuffle=True)
val_gen = LeafSequence(DATA / "val", IMG_SIZE, BATCH, augment=False, shuffle=False)
NUM_CLASSES = train_gen.num_classes
CLASS_NAMES = [k for k, _ in sorted(train_gen.class_indices.items(), key=lambda kv: kv[1])]
print(f"Train: {train_gen.n} | Val: {val_gen.n} | Clases: {CLASS_NAMES}")
with open(OUT / "class_indices_model2_pathogen.json", "w") as f:
    json.dump(train_gen.class_indices, f)

In [ ]:
import tensorflow as tf
import numpy as np

IMG_SIZE = globals().get("IMG_SIZE", (224, 224))
LR_P1 = globals().get("LR_P1", 1e-3)
LABEL_SMOOTH = globals().get("LABEL_SMOOTH", 0.05)
EMA_DECAY = globals().get("EMA_DECAY", 0.9999)
NUM_CLASSES = globals().get("NUM_CLASSES", 5)
CLASS_NAMES = globals().get("CLASS_NAMES", [])


class EMACallback(tf.keras.callbacks.Callback):
    def __init__(self, decay=0.9999):
        super().__init__()
        self.decay = decay
        self.ema_weights = None

    def on_train_begin(self, logs=None):
        self.ema_weights = [w.numpy().copy() for w in self.model.trainable_weights]

    def on_train_batch_end(self, batch, logs=None):
        for i, w in enumerate(self.model.trainable_weights):
            self.ema_weights[i] = self.decay * self.ema_weights[i] + (1 - self.decay) * w.numpy()

    def on_train_end(self, logs=None):
        for w, ew in zip(self.model.trainable_weights, self.ema_weights):
            w.assign(ew)


def build_pathogen_model(num_classes: int) -> tf.keras.Model:
    base = tf.keras.applications.EfficientNetB0(
        weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3)
    )
    base.trainable = False
    x = base.output
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(
        256, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(5e-5)
    )(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    output = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(base.input, output, name="model2_pathogen")


_counts = np.bincount(train_gen.labels, minlength=NUM_CLASSES)
_weights = _counts.sum() / (NUM_CLASSES * np.clip(_counts, 1, None))
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(_weights)}
print("class_weight M2:", dict(zip(CLASS_NAMES, [round(w, 3) for w in _weights])))

LOSS = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH)
METRICS = [
    "accuracy",
    tf.keras.metrics.Precision(name="prec"),
    tf.keras.metrics.Recall(name="rec"),
]

model = build_pathogen_model(NUM_CLASSES)
model.compile(optimizer=tf.keras.optimizers.Adam(LR_P1), loss=LOSS, metrics=METRICS)
model.summary()

In [ ]:
from pathlib import Path
import tensorflow as tf

EPOCHS_P1 = globals().get("EPOCHS_P1", 20)
OUT = globals().get("OUT", Path("./outputs"))

callbacks_p1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", mode="max", patience=6,
        restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(OUT / "model2_pathogen_p1_best.keras"),
        monitor="val_accuracy", mode="max", save_best_only=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1,
    ),
]

print("=== FASE 1: cabeza congelada ===")
h1 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_P1, callbacks=callbacks_p1,
    class_weight=CLASS_WEIGHT, verbose=1,
)

In [ ]:
from pathlib import Path
import numpy as np
import tensorflow as tf

EPOCHS_P2 = globals().get("EPOCHS_P2", 45)
LR_P2 = globals().get("LR_P2", 1e-5)
EMA_DECAY = globals().get("EMA_DECAY", 0.9999)
BATCH = globals().get("BATCH", 16)
OUT = globals().get("OUT", Path("./outputs"))

_h1 = globals().get("h1", None)
_initial_epoch = len(_h1.history["loss"]) if _h1 is not None else 0

for layer in model.layers:
    layer.trainable = True
for layer in model.layers[:-80]:
    layer.trainable = False
for layer in model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(1 for l in model.layers if l.trainable)
print(f"Layers entrenables en phase 2: {trainable_count}")

_steps = max(1, train_gen.n // BATCH) * EPOCHS_P2
_lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=LR_P2,
    decay_steps=_steps,
    alpha=0.01,
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(_lr_schedule),
    loss=LOSS,
    metrics=METRICS,
)

callbacks_p2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", mode="max", patience=12,
        restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(OUT / "model2_pathogen_p2_best.keras"),
        monitor="val_accuracy", mode="max", save_best_only=True, verbose=1,
    ),
    EMACallback(decay=EMA_DECAY),
]

print("=== FASE 2: fine-tuning ultimos 80 layers + Cosine LR + EMA ===")
h2 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=_initial_epoch + EPOCHS_P2, initial_epoch=_initial_epoch,
    callbacks=callbacks_p2, class_weight=CLASS_WEIGHT, verbose=1,
)
model.save(OUT / "model2_pathogen.keras")
print("Modelo guardado en", OUT / "model2_pathogen.keras")

## Evaluación en validación

Single-label: la predicción es `argmax` del softmax, sin thresholds por clase. Se reporta accuracy global, F1 por clase y F1 macro sobre el conjunto de validación. La evaluación independiente sobre el test set está en el notebook 05.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report


def collect_predictions(generator, model):
    y_true, y_pred = [], []
    for i in range(len(generator)):
        x_batch, y_batch = generator[i]
        y_true.append(np.argmax(y_batch, axis=1))
        y_pred.append(np.argmax(model.predict(x_batch, verbose=0), axis=1))
    return np.concatenate(y_true), np.concatenate(y_pred)


y_true_val, y_pred_val = collect_predictions(val_gen, model)

accuracy = accuracy_score(y_true_val, y_pred_val)
f1_per_class = f1_score(y_true_val, y_pred_val, average=None, labels=range(NUM_CLASSES), zero_division=0)
f1_macro = f1_score(y_true_val, y_pred_val, average="macro", zero_division=0)

report = {
    "accuracy": round(float(accuracy), 4),
    "f1_macro": round(float(f1_macro), 4),
    "f1_per_class": {name: round(float(f1_per_class[i]), 4) for i, name in enumerate(CLASS_NAMES)},
}
for name, value in report["f1_per_class"].items():
    print(f"{name:20s}  F1={value:.3f}")
print(f"\naccuracy={accuracy:.4f}  |  F1 macro={f1_macro:.4f}")
print(classification_report(y_true_val, y_pred_val, target_names=CLASS_NAMES, zero_division=0))

with open(OUT / "calibration_report.json", "w") as f:
    json.dump(report, f, indent=2)

In [ ]:
def diagnose_bias_variance_m2(h1, h2, model_name='M2'):
    all_loss  = h1.history['loss'] + h2.history['loss']
    all_vloss = h1.history['val_loss'] + h2.history['val_loss']
    all_acc   = h1.history['accuracy'] + h2.history['accuracy']
    all_vacc  = h1.history['val_accuracy'] + h2.history['val_accuracy']

    train_loss = all_loss[-1]
    val_loss   = all_vloss[-1]
    gap        = val_loss - train_loss
    val_acc    = all_vacc[-1]

    print(f'[{model_name}] train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | gap={gap:.4f}')
    print(f'[{model_name}] val_accuracy={val_acc:.4f} (objetivo >= 0.95)')
    if train_loss > 1.0:
        print('  ALTO BIAS — underfitting. Mas epocas, mas capacidad, menos regularizacion.')
    elif gap > 0.15:
        print('  ALTA VARIANZA — overfitting. Mas dropout, L2, mas augmentacion.')
    else:
        print('  Bias/varianza balanceados.')


_h1 = globals().get('h1', None)
_h2 = globals().get('h2', None)
if _h1 is not None and _h2 is not None:
    diagnose_bias_variance_m2(_h1, _h2, 'M2')

In [ ]:
import matplotlib.pyplot as plt

_h1 = globals().get("h1", None)
_h2 = globals().get("h2", None)
if _h1 is None or _h2 is None:
    print("Ejecuta primero las celdas de entrenamiento (FASE 1 y FASE 2)")
else:
    loss_h = _h1.history["loss"] + _h2.history["loss"]
    vloss = _h1.history["val_loss"] + _h2.history["val_loss"]
    acc = _h1.history["accuracy"] + _h2.history["accuracy"]
    vacc = _h1.history["val_accuracy"] + _h2.history["val_accuracy"]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ep = range(1, len(loss_h) + 1)
    div = len(_h1.history["loss"])
    ax1.plot(ep, loss_h, "b-", label="train")
    ax1.plot(ep, vloss, "r-", label="val")
    ax1.axvline(div, color="gray", linestyle="--", label="fine-tune")
    ax1.set_xlabel("Epoca"); ax1.set_ylabel("Cross-entropy"); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(ep, acc, "b-", label="train")
    ax2.plot(ep, vacc, "r-", label="val")
    ax2.axvline(div, color="gray", linestyle="--")
    ax2.set_xlabel("Epoca"); ax2.set_ylabel("Accuracy"); ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout()
    OUT = globals().get("OUT", __import__("pathlib").Path("./outputs"))
    plt.savefig(OUT / "model2_pathogen_curves.png", dpi=120)
    plt.show()